# Saturation level of each channel over time

*Draft prepared with Claude Code for A. Villano, 2026-11-02.*

Does the pulse height at which a channel saturates change during Run 76? For each of the six channels of the Side-2 readout, the **pulse height** (`current_max` of `goldsluice`'s `peak`: the largest sample above the baseline, in ADC counts) is drawn dump by dump along the run, in a long horizontal plot, one row per channel. A saturated channel stops at its own ceiling, so the **upper envelope** of the heights is the saturation level; a level that moves is a change of the gain or the bias of the channel.

**The data.** The eight DCRC1pB-triggered series near -82 V (-82.2 to -81.6 V; trigger `DCRC1pB 10`, 436 dumps), processed by `r76-82v-dcrc1pb.dataset.toml` (output in `~/goldsluice-output/r76-82v-dcrc1pb`). The voltage and trigger of every series are from `series.csv` of the processing folder. The series are in order of their start time (the eight digits after `0722` of a series name are month, day, hour and minute of 2022).

**The time axis.** One dump is about one minute of data (the duration of a series in minutes is its number of dumps), so each dump is placed at `series start + (dump - 1)` minutes. The weeks between the series are **left out**: the series follow each other and are separated by a vertical line, labelled with the date, the high voltage and the trigger. Real triggers only (`Unknown`; see `integral-on-the-face.ipynb`).

Run with the `darkmatter_cli_env` kernel.

In [ ]:
import csv
from datetime import datetime, timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

from goldpan.dataset import Dataset

LETTERS = "ABCDEF"

def field(tab, k, name):
    # one quantity of one channel: the channel is a struct column of the Parquet file
    return pc.struct_field(tab[f"Side2p{k}"].combine_chunks(), name).to_numpy(zero_copy_only=False)

_here = Path.cwd().resolve()
PROCESSING = next(p / "data_analysis" / "R76" / "processing" for p in [_here, *_here.parents] if (p / "data_analysis" / "R76" / "processing").is_dir())
DEFINITIONS = ["r76-82v-dcrc1pb.dataset.toml"]
OUTPUTS = [Path(Dataset.open(PROCESSING / d).spec.output) for d in DEFINITIONS]
CACHE = Path.home() / "goldsluice-output" / "relcal-height-per-dump-82v.npz"

with open(PROCESSING / "series.csv", newline="") as fh:
    meta = {r["series"]: r for r in csv.DictReader(fh)}

## Heights per dump

For every dump and channel: the 50th, 99th and 99.9th percentile of the height of the real triggers, and their maximum. The percentiles are used rather than the maximum alone because a few events have a glitch of several thousand counts (see `integral-vs-height.ipynb`). Reading the 436 files takes a minute or two the first time; the result is cached in `~/goldsluice-output/relcal-height-per-dump-82v.npz`.

In [ ]:
QS = [50, 99, 99.9]
if CACHE.exists():
    z = np.load(CACHE, allow_pickle=False)
    series_d, dump_d, nev, stat = z["series"], z["dump"], z["nev"], z["stat"]
else:
    S_, D_, N_, ST_ = [], [], [], []
    cols = ["trigger_type"] + [f"Side2p{k}" for k in LETTERS]
    for out in OUTPUTS:
        for f in sorted((out / "NAI-S2").glob("*_F[0-9][0-9][0-9][0-9].parquet")):
            tab = pq.read_table(f, columns=cols)
            real = np.asarray(tab["trigger_type"].to_pylist()) == "Unknown"
            h = np.column_stack([field(tab, k, "current_max") for k in LETTERS])[real]
            s, d = f.stem.rsplit("_F", 1)
            S_.append(s); D_.append(int(d)); N_.append(len(h))
            ST_.append(np.full((6, 4), np.nan) if len(h) < 20 else
                       np.column_stack([np.percentile(h, QS, axis=0).T, h.max(axis=0)]))   # (channel, [p50, p99, p99.9, max])
    series_d, dump_d, nev, stat = np.array(S_), np.array(D_), np.array(N_), np.array(ST_)
    np.savez(CACHE, series=series_d, dump=dump_d, nev=nev, stat=stat)
order = np.lexsort((dump_d, series_d))
series_d, dump_d, nev, stat = series_d[order], dump_d[order], nev[order], stat[order]
print(f"{len(dump_d):,} dumps in {len(np.unique(series_d))} series; real triggers per dump: median {int(np.median(nev))}, "
      f"{(nev < 20).sum()} dumps with fewer than 20 (left blank)")

## The run, channel by channel

Per dump: the 99th percentile of the height (line) and the maximum (faint dots). The vertical scale ends just above the plateau, so the few glitches of several thousand counts (the faint dots) are cut off. The series are shaded alternately; the labels give the start (month-day hour:minute of 2022), the high voltage and the trigger.

In [ ]:
x = np.zeros(len(dump_d))              # position along the plot: the dumps in order, in minutes, series one after the other
bounds, starts = [], []
pos = 0.0
for s in np.unique(series_d):
    m = series_d == s
    x[m] = pos + dump_d[m] - 1
    bounds.append((pos, pos + dump_d[m].max()))
    starts.append(s)
    pos += dump_d[m].max() + 20           # a gap of 20 minutes between series, for the separating line

def label(s):
    when = datetime(2022, int(s[4:6]), int(s[6:8]), int(s[9:11]), int(s[11:13]))
    r = meta.get(s, {})
    hv = f"{float(r['hv_v']):.1f} V" if r else "?"
    return f"{when:%m-%d %H:%M}\n{hv}\n{r.get('trigger', '?')}"

fig, axes = plt.subplots(6, 1, figsize=(26, 17), sharex=True)
for i, (ax, k) in enumerate(zip(axes, LETTERS)):
    for j, (a, b) in enumerate(bounds):
        if j % 2 == 0:
            ax.axvspan(a - 10, b + 10, color="0.93", zorder=0)
    ax.plot(x, stat[:, i, 3], ".", ms=1.5, color="0.55", zorder=2)
    for s_ in starts:                                              # one line per series: none across the gaps between them
        m_ = series_d == s_
        ax.plot(x[m_], stat[m_, i, 1], "-", lw=0.8, color="C0", zorder=3)
    ax.set_ylabel(f"Side2p{k}\nheight (counts)")
    ax.set_ylim(0, 1.3 * np.nanmax(stat[:, i, 1]))        # the plateau; the glitches of several thousand counts above it are cut off
    ax.set_xlim(-30, pos)
for (a, b), s in zip(bounds, starts):
    axes[0].text((a + b) / 2, 1.02, label(s), transform=axes[0].get_xaxis_transform(), ha="center", va="bottom", fontsize=7)
axes[-1].set_xlabel("time along the run (minutes of data; the gaps between series are left out)")
axes[-1].set_xticks([])
fig.tight_layout(rect=(0, 0, 1, 0.95))
plt.show()

## The saturation level by series

The 99.99th percentile of the height of the real triggers of each series (the same definition as in `integral-vs-height.ipynb`, there over the whole run), channel by channel, and the same relative to the median over the series. A saturated channel is flat at its ceiling, so a change of this number from one series to the next is a change of the level.

In [ ]:
# the 99.99th percentile needs the events, not the per-dump statistics: read them series by series
LEVEL_CACHE = Path.home() / "goldsluice-output" / "relcal-saturation-per-series-82v.npz"
sers = list(np.unique(series_d))
if LEVEL_CACHE.exists():
    level = np.load(LEVEL_CACHE)["level"]
else:
    level = np.full((len(sers), 6), np.nan)
    cols = ["trigger_type"] + [f"Side2p{k}" for k in LETTERS]
    for n, s in enumerate(sers):
        parts = []
        for out in OUTPUTS:
            files = sorted((out / "NAI-S2").glob(f"{s}_F[0-9][0-9][0-9][0-9].parquet"))
            if files:
                parts = [pq.read_table(f, columns=cols) for f in files]
                break
        tab = pa.concat_tables(parts)
        real = np.asarray(tab["trigger_type"].to_pylist()) == "Unknown"
        h = np.column_stack([field(tab, k, "current_max") for k in LETTERS])[real]
        level[n] = np.percentile(h, 99.99, axis=0)
    np.savez(LEVEL_CACHE, level=level)

print(f"{'series':>14} {'HV (V)':>7} {'trigger':>12}" + "".join(f"{'Side2p' + k:>10}" for k in LETTERS))
for s, row in zip(sers, level):
    r = meta[s]
    print(f"{s:>14} {float(r['hv_v']):>7.1f} {r['trigger']:>12}" + "".join(f"{v:>10.0f}" for v in row))
print(f"{'median':>35}" + "".join(f"{v:>10.0f}" for v in np.median(level, axis=0)))
print(f"{'largest change from median':>35}" + "".join(f"{np.max(np.abs(level[:, i] / np.median(level[:, i]) - 1)):>10.1%}" for i in range(6)))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
hv = np.array([float(meta[s]["hv_v"]) for s in sers])
for i, k in enumerate(LETTERS):
    ax1.plot(range(len(sers)), level[:, i] / np.median(level[:, i]), "o-", label=f"Side2p{k}")
    ax2.plot(hv, level[:, i] / np.median(level[:, i]), "o", label=f"Side2p{k}")
ax1.set_xticks(range(len(sers)))
ax1.set_xticklabels([s[4:] for s in sers], rotation=90, fontsize=7)
ax1.set_xlabel("series, in order of time (month day _ hour minute)")
ax2.set_xlabel("high voltage (V)")
for ax in (ax1, ax2):
    ax.set_ylabel("saturated height / its median over the series")
    ax.axhline(1, color="0.5", lw=0.5)
ax1.legend(ncol=3, fontsize=8)
fig.tight_layout()
plt.show()

## From a small pulse up to the distorted pulses, channel by channel

For every channel, a progression of real-trigger events whose `integral` of that channel rises from a small pulse to 15 % of the **ceiling integral** of the channel (`saturated height x 2.88 ms`; the saturated height is the median over the series of the 99.99th percentile above, 2.88 ms is what is left of the trace after the pulse starts). Beyond about 15 % the pulses are distorted, so the scan is fine below that: nine stages from 1 % to 15 %. Pulses are drawn as the height above the baseline, in ADC counts.

**Clean pulses only.** The pool of events is 14 dumps (two of each series; the series `07221113_1159`, low in every channel, is left out), read from the raw dumps in `~/idx`. A pulse of a channel is taken as clean when, in that channel's own trace:

* the pulse is above 6 times the noise of the baseline, and not saturated (below 1.05 times the saturated height);
* the baseline window is quiet: the smoothed pretrigger (samples 10 to 500, smoothed over 16 samples) stays within 5 times the smoothed noise of its mean, so there is no pulse before the trigger;
* the peak is in the first 100 samples after the trigger (the pulse starts at the trigger, so a larger pulse later in the trace, or a pile-up, is out);
* after the peak the smoothed trace never rises again by more than 4 % of the height (and 4 times the smoothed noise) above its lowest point so far: no second pulse, no step.

At each stage the event whose integral is closest to the stage is chosen from the 10 nearest clean ones with `SEED`. The six channels of a stage are different events. This is a selection by shape, so the pulses shown are the cleanest ones, not typical ones.

In [ ]:
import warnings

import goldpan.channels as pch
from goldpan.adapters import idx
from goldpan.config import DCRC_RUN76

warnings.filterwarnings("once")
CONFIG = DCRC_RUN76                                  # 0.8 us per sample, baseline = samples 10 to 500
RAW = Path.home() / "idx"
EXCLUDE = {"07221113_1159"}                           # the series that is low in every channel
FRACTIONS = [0.01, 0.02, 0.03, 0.045, 0.06, 0.08, 0.10, 0.125, 0.15]
DUMPS = (10, 30)                                      # the pool: these dumps of every series
N_NEAREST = 10
SEED = 1
N_SAMPLES = 4096
SENSORS = {c.name: (c.detector, c.channel) for c in pch.sensor_channels()}
g_, p_ = CONFIG.glitch_samples, CONFIG.pretrigger_samples
SM = 16                                               # samples of the moving average

SAT = np.median(level, axis=0)                        # saturated height of each channel, counts
CEILING = SAT * (N_SAMPLES - p_) * CONFIG.sample_period_s
print("saturated height (counts): " + ", ".join(f"{k} {h:.0f}" for k, h in zip(LETTERS, SAT)))
print("ceiling integral (counts x s): " + ", ".join(f"{k} {c:.2f}" for k, c in zip(LETTERS, CEILING)))

def smooth(y):
    c = np.cumsum(np.pad(y, ((0, 0), (1, 0))), axis=1)
    return (c[:, SM:] - c[:, :-SM]) / SM               # moving average, length n - SM + 1

def metrics(x, sat):
    # x: (events, samples) raw traces of one channel. Returns integral, height and whether the pulse is clean.
    base = x[:, g_:p_].mean(axis=1, keepdims=True)
    y = x - base
    sigma = x[:, g_:p_].std(axis=1)
    height = y.max(axis=1)
    peak = y.argmax(axis=1)
    ys = smooth(y)
    sig_s = sigma / np.sqrt(SM)
    pre = np.abs(ys[:, g_:p_ - SM]).max(axis=1)                       # the quiet pretrigger window
    t = np.arange(ys.shape[1])[None, :]
    pk = np.clip(peak - SM // 2, 0, ys.shape[1] - 1)
    z = np.where(t >= pk[:, None], ys, np.inf)
    rebound = np.where(t >= pk[:, None], ys - np.minimum.accumulate(z, axis=1), 0).max(axis=1)
    clean = ((height > 6 * sigma) & (height < 1.05 * sat) & (pre < 5 * sig_s + 0.01 * height)
             & (peak >= p_) & (peak < p_ + 100) & (rebound < np.maximum(0.04 * height, 4 * sig_s)))
    return y.sum(axis=1) * CONFIG.sample_period_s, height, clean

pool = []                                              # one dict per dump: series, dump, event, integral/height/clean (events x 6)
for s in sorted({str(s) for s in series_d} - EXCLUDE):
    for dn in DUMPS:
        folder = RAW / f"{s}_F{dn:04d}"
        e = idx.read(folder, detector_name="NAI-S2", channels=SENSORS, config=CONFIG)
        seg = {name: sg for (name, sg) in e.traces}
        integ, height, clean = [], [], []
        for i, k in enumerate(LETTERS):
            x = np.asarray(e.traces[(f"Side2p{k}", seg[f"Side2p{k}"])], dtype=np.float32)
            a, b, c = metrics(x, SAT[i])
            integ.append(a); height.append(b); clean.append(c)
        number = np.asarray(e.quantities["event"])
        tab = pq.read_table(next((o / "NAI-S2" / f"{s}_F{dn:04d}.parquet" for o in OUTPUTS if (o / "NAI-S2" / f"{s}_F{dn:04d}.parquet").exists())),
                            columns=["event", "trigger_type"])
        real = np.isin(number, tab["event"].to_numpy()[np.asarray(tab["trigger_type"].to_pylist()) == "Unknown"])
        pool.append(dict(series=s, dump=dn, event=number, real=real, integral=np.array(integ).T, height=np.array(height).T, clean=np.array(clean).T))
n_pool = sum(len(p["event"]) for p in pool)
print(f"{len(pool)} dumps, {n_pool:,} events in the pool; " + ", ".join(
    f"{k}: {sum((p['clean'][:, i] & p['real']).sum() for p in pool):,} clean" for i, k in enumerate(LETTERS)))

In [ ]:
rng = np.random.default_rng(SEED)
cat = {key: np.concatenate([p[key] for p in pool]) for key in ("event", "real", "integral", "height", "clean")}
cat["dumpref"] = np.concatenate([np.full(len(p["event"]), n) for n, p in enumerate(pool)])
chosen = {}                                            # channel index -> list of rows of cat, one per stage
for i, k in enumerate(LETTERS):
    ok = np.flatnonzero(cat["real"] & cat["clean"][:, i])
    picks = []
    for frac in FRACTIONS:
        near = ok[np.argsort(np.abs(cat["integral"][ok, i] - frac * CEILING[i]))[:N_NEAREST]]
        picks.append(int(rng.choice(near)))
    chosen[i] = picks
print("integral / height of the event chosen for each stage and channel")
print(f"{'stage':>7}" + "".join(f"{'Side2p' + k:>16}" for k in LETTERS))
for s, frac in enumerate(FRACTIONS):
    print(f"{frac:>6.1%} " + "".join(f"{cat['integral'][chosen[i][s], i]:>8.2f}/{cat['height'][chosen[i][s], i]:>6.0f} " for i in range(6)))

In [ ]:
# the traces of the chosen events: read the dumps that hold them once more
trace = {}                                             # row of cat -> (6, samples) raw traces
by_dump = {}
for i in range(6):
    for j in chosen[i]:
        by_dump.setdefault(int(cat["dumpref"][j]), set()).add(j)
offset = np.cumsum([0] + [len(p["event"]) for p in pool])
for n, rows in sorted(by_dump.items()):
    p = pool[n]
    e = idx.read(RAW / f"{p['series']}_F{p['dump']:04d}", detector_name="NAI-S2", channels=SENSORS, config=CONFIG)
    seg = {name: sg for (name, sg) in e.traces}
    for j in rows:
        r = j - offset[n]
        trace[j] = np.array([np.asarray(e.traces[(f"Side2p{k}", seg[f"Side2p{k}"])][r], dtype=float) for k in LETTERS])
t_ms = (np.arange(N_SAMPLES) - p_) * CONFIG.sample_period_s * 1e3

def flat(j, i):
    x = trace[j][i]
    return x - x[g_:p_].mean()

worst = max(abs(flat(j, i).sum() * CONFIG.sample_period_s - cat["integral"][j, i]) for i in range(6) for j in chosen[i])
print(f"{len({j for i in range(6) for j in chosen[i]})} traces read; largest difference from the integral computed in the pool: {worst:.1e}")

In [ ]:
nst = len(FRACTIONS)
fig, axes = plt.subplots(6, nst, figsize=(2.3 * nst, 11), sharex=True)
for i, k in enumerate(LETTERS):
    top = 1.15 * max(cat["height"][j, i] for j in chosen[i])             # one vertical scale per channel row, up to its tallest pulse
    for s, frac in enumerate(FRACTIONS):
        ax = axes[i, s]
        j = chosen[i][s]
        f = flat(j, i)
        ax.axvline(0, color="0.6", ls=":", lw=0.7)
        ax.fill_between(t_ms, f, 0, color="C0", alpha=0.2, lw=0)
        ax.plot(t_ms, f, lw=0.7, color="C0")
        ax.set_ylim(-0.08 * top, top)
        ax.set_title(f"integral {cat['integral'][j, i]:.2f}\nheight {cat['height'][j, i]:.0f}", fontsize=8)
        if s == 0:
            ax.set_ylabel(f"Side2p{k}\nheight above baseline (counts)")
        elif s == nst - 1:
            ax.yaxis.tick_right()
        else:
            ax.set_yticklabels([])
for s, frac in enumerate(FRACTIONS):
    axes[0, s].text(0.5, 1.4, f"{frac:.1%} of the ceiling", transform=axes[0, s].transAxes, ha="center", fontsize=9, weight="bold")
    axes[-1, s].set_xlabel("time from the trigger (ms)")
fig.tight_layout()
plt.show()

### The same as a movie

One frame per stage: the six channels side by side, each with its own clean event of that stage; the vertical scale of every channel is that of its tallest chosen pulse. Saved as `figures/saturation-progression.gif` next to this notebook.

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter

FIG_DIR = Path.cwd() / "figures"
FIG_DIR.mkdir(exist_ok=True)
fig, axes = plt.subplots(2, 3, figsize=(13, 6.5), sharex=True)
lines, texts = [], []
for ax, i, k in zip(axes.flat, range(6), LETTERS):
    top = 1.15 * max(cat["height"][j, i] for j in chosen[i])
    ax.axvline(0, color="0.6", ls=":", lw=0.7)
    ax.set_ylim(-0.08 * top, top)
    ax.set_title(f"Side2p{k}", fontsize=11)
    ax.set_xlim(t_ms[0], t_ms[-1])
    (ln,) = ax.plot([], [], lw=0.9, color="C0")
    texts.append(ax.text(0.97, 0.93, "", transform=ax.transAxes, ha="right", va="top", fontsize=9))
    lines.append(ln)
for ax in axes[-1]:
    ax.set_xlabel("time from the trigger (ms)")
for ax in axes[:, 0]:
    ax.set_ylabel("height above baseline (counts)")
head = fig.suptitle("", fontsize=12)

def frame(s):
    head.set_text(f"integral at {FRACTIONS[s]:.1%} of the ceiling")
    for i in range(6):
        j = chosen[i][s]
        lines[i].set_data(t_ms, flat(j, i))
        texts[i].set_text(f"integral {cat['integral'][j, i]:.2f}\nheight {cat['height'][j, i]:.0f}")
    return lines + texts

fig.tight_layout(rect=(0, 0, 1, 0.95))
anim = FuncAnimation(fig, frame, frames=len(FRACTIONS), interval=900, blit=False)
GIF = FIG_DIR / "saturation-progression.gif"
anim.save(GIF, writer=PillowWriter(fps=1.1), dpi=70)
plt.close(fig)
from IPython.display import Image
Image(filename=str(GIF))

**What this shows, and what is not done.** The height ceiling of every channel dump by dump over the eight -82 V series, and per series. Not done: the seven NaI-triggered series near -82 V (not processed yet) and an explanation of any step; the grid of pulses and the movie show what a pulse looks like on the way to saturation but not why the ceiling is where it is; a change in the level at the start of a series may be a change of the bias of the channel (the high voltage changes slightly between the series, -82.2 to -81.6 V) or of the readout, which this plot cannot tell apart. The ceiling of the integral is this height times the 2.88 ms that is left of the trace after the pulse starts (`spectrum-82v-total.ipynb`).